In [109]:
import numpy as np
from scipy.special import erf
import maps as mp
import scipy.optimize as opt
import matplotlib.pyplot as plt
import importlib
import time

importlib.reload(mp)

<module 'maps' from '/Users/lucaraffo/Desktop/STOCH/maps.py'>

In [110]:
# observation times
t_obs = np.arange(1, 6)          # [1,2,3,4,5]

# observed noisy data
y_obs= np.array([0.18, 0.32, 0.42, 0.49, 0.54])

# noise variance sigma^2
sigma2 = 1e-3

In [111]:
def a_ox(x1):
    return 0.4 + 0.4 * (1.0 + erf(x1 / np.sqrt(2.0)))

def b_ox(x2):
    return 0.01 + 0.15 * (1.0 + erf(x2 / np.sqrt(2.0)))

def B_model(t, x):
    x1 = x[:, 0]
    x2 = x[:, 1]
    a = a_ox(x1)
    b = b_ox(x2)
    out = a[:, None] * (1.0 - np.exp(-b[:, None] * t[None, :]))
    if x.ndim == 1:
        return out[0]
    else:
        return out

def log_prior(x):
    lp = -0.5 * np.sum(x * x, axis = 1)
    if x.ndim == 1:
        return lp[0]
    else:
        return lp

def log_likelihood(x):
    pred = B_model(t_obs, x)
    if pred.ndim == 1:
        r = y_obs - pred
        return -0.5 * np.sum(r * r) / sigma2
    r = y_obs[None, :] - pred
    return -0.5 * np.sum(r * r, axis = 1) / sigma2

def log_posterior(x):
    return log_likelihood(x) + log_prior(x)


In [10]:
# create degree-d map
d = 1
T1 = mp.TriangularTransport2D(degree = 1)

# initial parameters near identity
thetaT_1 = T1.initial_theta()

# fixed number of samples from prior
M = 2000
X = np.random.randn(M, 2)

def neg_log_KL(theta):
    """ monte carlo approximation of eq. (9), up to an additive constant """
    Z = T1.forward(X, theta)                 # pushforward 
    log_pi = log_posterior(Z)              # log pi^y(z)
    log_det = T1.log_det_jacobian(X, theta)

    # minus sign
    return np.mean(-log_pi - log_det)

# run optimization
succ = False
run = 100
while succ is False:
    run *= 2
    time_init = time.time()
    res1 = opt.minimize(neg_log_KL, thetaT_1, method = "L-BFGS-B", options = {"maxiter": run})
    time_fin = time.time()
    succ = res1.success
timeT1 = time_fin - time_init

In [ ]:
np.save("thetaT1.npy", res1.x)

In [ ]:
# create degree-d map
d = 2
T2 = mp.TriangularTransport2D(degree = d)

# initial parameters near identity
thetaT_2 = T2.initial_theta()

# fixed number of samples from prior
M = 2000
X = np.random.randn(M, 2)

def neg_log_KL(theta):
    """ monte carlo approximation of eq. (9), up to an additive constant """
    Z = T2.forward(X, theta)                 # pushforward 
    log_pi = log_posterior(Z)              # log pi^y(z)
    log_det = T2.log_det_jacobian(X, theta)

    # minus sign
    return np.mean(-log_pi - log_det)

# run optimization
succ = False
run = 100
while succ is False:
    run *= 2
    time_init = time.time()
    res2 = opt.minimize(neg_log_KL, thetaT_2, method = "L-BFGS-B", options = {"maxiter": run})
    time_fin = time.time()
    succ = res2.success
timeT2 = time_fin - time_init

In [ ]:
np.save("thetaT2.npy", res2.x)

In [ ]:
# create degree-d map
d = 3
T3 = mp.TriangularTransport2D(degree = d)

# initial parameters near identity
thetaT_3 = T3.initial_theta()

# fixed number of samples from prior
M = 2000
X = np.random.randn(M, 2)

def neg_log_KL(theta):
    """ monte carlo approximation of eq. (9), up to an additive constant """
    Z = T3.forward(X, theta)                 # pushforward 
    log_pi = log_posterior(Z)              # log pi^y(z)
    log_det = T3.log_det_jacobian(X, theta)

    # minus sign
    return np.mean(-log_pi - log_det)

# run optimization
succ = False
run = 100
while succ is False:
    run *= 2
    time_init = time.time()
    res3 = opt.minimize(neg_log_KL, thetaT_3, method = "L-BFGS-B", options = {"maxiter": run})
    time_fin = time.time()
    succ = res3.success
timeT3 = time_fin - time_init

In [ ]:
np.save("thetaT3.npy", res3.x)

In [ ]:
# create degree-d map
d = 4
T4 = mp.TriangularTransport2D(degree = d)

# initial parameters near identity
thetaT_4 = T4.initial_theta()

# fixed number of samples from prior
M = 2000
X = np.random.randn(M, 2)

def neg_log_KL(theta):
    """ monte carlo approximation of eq. (9), up to an additive constant """
    Z = T4.forward(X, theta)                 # pushforward 
    log_pi = log_posterior(Z)              # log pi^y(z)
    log_det = T4.log_det_jacobian(X, theta)

    # minus sign
    return np.mean(-log_pi - log_det)

# run optimization
succ = False
run = 100
while succ is False:
    run *= 2
    time_init = time.time()
    res4 = opt.minimize(neg_log_KL, thetaT_4, method = "L-BFGS-B", options = {"maxiter": run})
    time_fin = time.time()
    succ = res4.success
timeT4 = time_fin - time_init

In [ ]:
np.save("thetaT4.npy", res4.x)

In [117]:
posterior_samples = np.load("posterior_samples.npy")
samples_training = posterior_samples

In [ ]:
# degree-1 inverse map S1
d = 1
S1 = mp.TriangularTransport2D(degree = d)

thetaS_1 = S1.initial_theta()

Z = samples_training.copy()

def neg_log_KL_S1(theta):
    Y = S1.forward(Z, theta)
    log_phi = log_prior(Y)
    log_det = S1.log_det_jacobian(Z, theta)
    return np.mean(-log_phi - log_det)

succ = False
run = 2000
time_init = time.time()
res1_inv = opt.minimize(neg_log_KL_S1, thetaS_1, method = "L-BFGS-B", options = {"maxiter": run})
time_fin = time.time()
timeS1 = time_fin - time_init

In [89]:
np.save("thetaS1.npy", res1_inv.x)

In [90]:
# degree-2 inverse map S2
d = 2
S2 = mp.TriangularTransport2D(degree = d)

thetaS_2 = S2.initial_theta()

Z = samples_training.copy()

def neg_log_KL_S2(theta):
    Y = S2.forward(Z, theta)
    log_phi = log_prior(Y)
    log_det = S2.log_det_jacobian(Z, theta)
    return np.mean(-log_phi - log_det)

succ = False
run = 2000
time_init = time.time()
res2_inv = opt.minimize(neg_log_KL_S2, thetaS_2, method = "L-BFGS-B", options = {"maxiter": run})
time_fin = time.time()
timeS2 = time_fin - time_init

In [91]:
np.save("thetaS2.npy", res2_inv.x)

In [92]:
# degree-3 inverse map S3
d = 3
S3 = mp.TriangularTransport2D(degree = d)

thetaS_3 = S3.initial_theta()

Z = samples_training.copy()

def neg_log_KL_S3(theta):
    Y = S3.forward(Z, theta)
    log_phi = log_prior(Y)
    log_det = S3.log_det_jacobian(Z, theta)
    return np.mean(-log_phi - log_det)

succ = False
run = 2000
time_init = time.time()
res3_inv = opt.minimize(neg_log_KL_S3, thetaS_3, method = "L-BFGS-B", options = {"maxiter": run})
time_fin = time.time()
timeS3 = time_fin - time_init

In [93]:
np.save("thetaS3.npy", res3_inv.x)

In [94]:
# degree-4 inverse map S4
d = 4
S4 = mp.TriangularTransport2D(degree = d)

thetaS_4 = S4.initial_theta()

Z = samples_training.copy()

def neg_log_KL_S4(theta):
    Y = S4.forward(Z, theta)
    log_phi = log_prior(Y)
    log_det = S4.log_det_jacobian(Z, theta)
    return np.mean(-log_phi - log_det)

succ = False
run = 2000
time_init = time.time()
res4_inv = opt.minimize(neg_log_KL_S4, thetaS_4, method = "L-BFGS-B", options = {"maxiter": run})
time_fin = time.time()
timeS4 = time_fin - time_init

In [95]:
np.save("thetaS4.npy", res4_inv.x)